# 02 · 2D detector (zero-shot, COCO weights)

Runs torchvision Faster R-CNN v2 with COCO weights, never fine-tuned (the main configuration adds no KITTI labels), on:

- **KITTI 3D Object**: the 7,481 training images. These detections choose MTrans's 500 labelled frames and give MTrans its input boxes on every other frame;
- **KITTI-360 paper drives**: usable frames every 2.5 m, plus every frame the student uses (sampled every 5 m; on drive 0000, which stores every frame, the two picks differ).

Output per source: `<dataset>/<sequence>.txt` in our classes (car, large_vehicle, pedestrian, cyclist) and `<sequence>_raw.txt` in COCO classes, one detection per line: `fid cls score l t r b`. A cyclist is a person merged with the bicycle/motorcycle under them.

## Before you run

1. **Settings → Accelerator:** GPU T4 x2 (one is used). **Internet:** On (code clone and the ~170 MB COCO weights).
2. **Add Input:** `garymk/kitti-3d-object-detection-dataset`; the KITTI-360 starter (drive 0000 images and calibration); `data-poses-zip`; each `kitti360-d<drive>-supplement` you have, plus drive 0000's data-prep output. Drives without an attached supplement are skipped and listed.
3. Trial: `MAX_IMAGES = 50`. Real run: `MAX_IMAGES = None`, then **Save Version → Save & Run All** (about 1 to 1.5 h for ~30k images).

In [ ]:
from pathlib import Path
import time

REPO_URL    = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF     = "p0/skeleton-geometry"     # branch, tag or commit (main once merged); pin a commit for the final run
INPUT_ROOT  = Path("/kaggle/input")
OUT_DIR     = Path("/kaggle/working/detector2d_v1")   # kept as the notebook output
SCRATCH     = Path("/kaggle/temp")
RUN_KITTI   = True
DRIVES      = ["0000", "0002", "0004", "0005", "0006", "0009", "0010"]   # paper drives (kitti360_paper_v1)
EVERY_M     = 2.5
BATCH_SIZE  = 8
MIN_SIZE, MAX_SIZE = 800, 2400          # upscales KITTI/KITTI-360 images ~1.7-1.9x for small, distant objects
SCORE_FLOOR = 0.05                      # low floor: stricter thresholds are applied by whoever reads the files
MAX_IMAGES  = None                      # e.g. 50 per source for a trial

CODE_DIR = SCRATCH / "OCGF"
SUMMARY, ITEMS = {}, {}

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import shutil, subprocess, sys

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

SCRATCH.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))      # the editable install is only seen by new processes
# a rerun in the same session must not keep using modules imported from an earlier clone
for name in [m for m in sys.modules if m == "ocgf" or m.startswith("ocgf.")]:
    del sys.modules[name]

import torch
from ocgf.utils.provenance import environment
print(environment())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")

## Inputs

KITTI's `training/image_2` and, per drive, its data-prep output plus the raw inputs it needs.

In [ ]:
from ocgf.datasets.kitti360 import Kitti360Sequence
from ocgf.datasets.locate import find_kitti_object, prepared_drive

kitti = find_kitti_object(INPUT_ROOT) if RUN_KITTI else {}
if RUN_KITTI:
    print("KITTI training/image_2:", kitti.get("image_2", "NOT FOUND"))
drives = {}
for d in DRIVES:
    seq = f"2013_05_28_drive_{d}_sync"
    try:
        drives[d] = Kitti360Sequence(sequence=seq, components=prepared_drive(INPUT_ROOT, seq))
        source = "image shards" if "image_shards" in drives[d].paths else "starter images"
        print(f"drive {d}: ready ({source})")
    except FileNotFoundError as err:
        print(f"drive {d}: skipped, {err}")
assert not RUN_KITTI or "image_2" in kitti, "KITTI training/image_2 not found: attach the KITTI 3D object dataset"
assert drives or RUN_KITTI, 'nothing to run on'

## Detector

In [ ]:
from ocgf.teachers.detector2d.model import Detector

detector = Detector(min_size=MIN_SIZE, max_size=MAX_SIZE, score_floor=SCORE_FLOOR)
print(detector.device, detector.settings)

## KITTI 3D Object

In [ ]:
from ocgf.teachers.detector2d.run import kitti360_frames, kitti_object_items, load_png, run_source

if RUN_KITTI:
    items = kitti_object_items(kitti["image_2"])[:MAX_IMAGES]
    ITEMS["kitti/training"] = (items, load_png)
    SUMMARY["kitti/training"] = run_source(detector, items, load_png, OUT_DIR, "kitti", "training", BATCH_SIZE)
    print(SUMMARY["kitti/training"])

## KITTI-360 drives

In [ ]:
for d, seq in drives.items():
    frames = kitti360_frames(seq, EVERY_M)[:MAX_IMAGES]
    items = [(seq.fid(int(f)), int(f)) for f in frames]
    key = f"kitti360/{seq.sequence}"
    ITEMS[key] = (items, seq.image)
    SUMMARY[key] = run_source(detector, items, seq.image, OUT_DIR, "kitti360", seq.sequence, BATCH_SIZE)
    print(d, SUMMARY[key])

## Examples

Three images per source with their mapped detections (score ≥ 0.3).

In [ ]:
from IPython.display import Image, display
from ocgf.teachers.detector2d.mapping import Detections, read_detections
from ocgf.teachers.detector2d.viz import draw_detections

for key, (items, load) in ITEMS.items():
    dets = read_detections(OUT_DIR / f"{key}.txt")
    for fid, ref in items[:: max(1, len(items) // 3)][:3]:
        png = draw_detections(load(ref), dets.get(fid, Detections.empty()), OUT_DIR / "examples" / f"{fid.replace('/', '_')}.png", fid)
        display(Image(filename=str(png)))

## Manifest

In [ ]:
from ocgf.teachers.detector2d.mapping import COCO_TO_OURS
from ocgf.utils.provenance import write_manifest

write_manifest(OUT_DIR / "manifest.json", teacher="detector2d", settings=detector.settings,
               weights_sha256=detector.weights_sha256(), mapping=COCO_TO_OURS, every_m=EVERY_M, sources=SUMMARY)
for key, s in SUMMARY.items():
    print(f"{key:45s} {s['images']:6d} images  {s['seconds'] / 60:5.1f} min  {s['detections']}")
sh(f'du -sh "{OUT_DIR}"/* | sort -h')

## After the run

- Turn the output into the private dataset `ocgf-detector2d-v1` (Output tab → New Dataset); the MTrans and student notebooks attach it.
- Download `detector2d_v1/kitti/training.txt` (a few MB) to `data/detector2d_v1/kitti/training.txt` locally and run
  `python scripts/select_labelled_frames.py --detections data/detector2d_v1/kitti/training.txt`: it writes the frozen 500/125 labelled-frame lists for MTrans.